[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检 Environment Check

<span style="background:#1a3a5c;color:#7ec8ff;padding:2px 10px;border-radius:10px;font-size:.85em">CPU</span>
配套讲解：[`00_overview.html`](00_overview.html) · 课程主页：[`../index.html`](../index.html)

本 notebook 做四件事，全部 **纯 CPU、不下载模型、几秒跑完**：

1. **依赖自检** —— 缺什么包一眼看到（标红）；
2. **运行时检测** —— device（cuda / mps / cpu）与 API key 是否配置（只看 bool，不打印 key）；
3. **subprocess 沙箱冒烟测试** —— 本课所有 agent 执行代码的安全地基；
4. **最小 tool registry** —— agent "手" 的雏形，模块 01 会展开成 JSON-schema 版。

最后是 2 道 **✏️ 练习**（必做）：你将亲手写出贯穿全课的两个轮子 —— `run_sandboxed` 和 `dispatch`。
自测 cell 的 `assert` 全部通过即达标；文末有 📖 参考答案，先自己做再对照。

> kernel 选择 **Agents Course**（没有就先跑 `python -m ipykernel install --user --name agents --display-name "Agents Course"`）。

In [1]:
# ── 依赖自检：逐个 try/except 导入，缺的标红 ──
import importlib, sys

RED, GREEN, RESET = "\033[91m", "\033[92m", "\033[0m"
print(f"Python {sys.version.split()[0]}  (建议 >= 3.10)\n")

packages = ["torch", "transformers", "openai", "anthropic",
            "numpy", "pandas", "PIL", "pytest"]
missing = []
for name in packages:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "?")
        print(f"  {GREEN}✓{RESET} {name:<14} {ver}")
    except ImportError:
        missing.append(name)
        print(f"  {RED}✗ {name:<14} 未安装{RESET}")

if missing:
    print(f"\n{RED}缺少 {len(missing)} 个包: {missing}{RESET}")
    print("修复: pip install -r ../requirements.txt   (注: PIL 由 pillow 提供)")
else:
    print(f"\n{GREEN}✅ 全部依赖就绪{RESET}")

Python 3.11.15  (建议 >= 3.10)

  ✓ torch          2.6.0+cu124
  ✓ transformers   5.11.0
  ✓ openai         2.41.1
  ✓ anthropic      0.109.1
  ✓ numpy          2.4.4
  ✓ pandas         3.0.3
  ✓ PIL            12.2.0
  ✓ pytest         9.0.3

✅ 全部依赖就绪


In [2]:
# ── 运行时检测：device 与 API key ──
import os

# device：本课 CPU-first，GPU / MPS 只是可选加速
try:
    import torch
    if torch.cuda.is_available():
        device, note = "cuda", torch.cuda.get_device_name(0)
    elif torch.backends.mps.is_available():
        device, note = "mps", "Apple Silicon"
    else:
        device, note = "cpu", "本课核心内容 CPU 全部可跑"
    print(f"device = {device}  ({note})")
except ImportError:
    print("torch 未安装，跳过 device 检测 —— 先完成上一个 cell 的依赖安装")

# API key：只打印是否存在（bool），绝不打印 key 的值
print()
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]:
    print(f"{key:<18} configured = {bool(os.environ.get(key))}")
print("\n没有 key 也完全 OK：课程默认用本地 Qwen2.5-1.5B-Instruct 当 agent 大脑，"
      "有 key 时各 notebook 会自动切换到闭源模型（体验更佳）。")

device = cuda  (NVIDIA GeForce RTX 4090)

OPENAI_API_KEY     configured = False
ANTHROPIC_API_KEY  configured = False

没有 key 也完全 OK：课程默认用本地 Qwen2.5-1.5B-Instruct 当 agent 大脑，有 key 时各 notebook 会自动切换到闭源模型（体验更佳）。


## 两个贯穿全课的最小机制

**① subprocess 沙箱。** agent 要执行代码 / shell 命令，但绝不能直接跑在你的解释器里 ——
失控的命令（死循环、误删文件）必须能被隔离和杀掉。本课的做法贯穿始终：
一切执行走 `subprocess.run(..., capture_output=True, timeout=...)`，
**`timeout` 是 harness 的安全带**：超时即杀进程，绝不拖死整个评测。模块 03 会把它升级成带资源限制与轨迹记录的完整 harness。

**② tool registry。** agent 的"手"本质上就是一张表：`工具名 → 可调用对象`。
LLM 输出工具名和参数，框架查表执行，把结果回灌上下文。下面先看 5 行代码的雏形 —— 模块 01 把它升级成带 JSON-schema 校验的正式版。

In [3]:
# ── 沙箱冒烟测试 ──
import subprocess, sys

# 测试 1：最普通的命令，capture_output 拿到输出，timeout 兜底
p = subprocess.run(["echo", "hello"], capture_output=True, text=True, timeout=5)
print("stdout     :", p.stdout.strip())
print("returncode :", p.returncode)

# 测试 2：timeout 参数生效 —— 子进程要睡 10 秒，我们只给 1 秒
# (用 sys.executable -c 而不是 shell 的 sleep，跨平台且不依赖外部命令)
try:
    subprocess.run([sys.executable, "-c", "import time; time.sleep(10)"],
                   capture_output=True, timeout=1)
    print("不应该到这里")
except subprocess.TimeoutExpired:
    print("✓ TimeoutExpired 如期抛出 —— 失控的命令会被杀掉，不会拖死 harness")

FileNotFoundError: [WinError 2] 系统找不到指定的文件。

In [4]:
# ── 最小 tool registry：dict 注册，按名字分发 ──

NOW_FIXED = "2026-01-01 00:00:00 (固定字符串，保证 notebook 可复现)"

def add(a, b):
    '''两数相加。'''
    return a + b

def now():
    '''返回固定的时间字符串（真实 agent 里这会读系统时钟）。'''
    return NOW_FIXED

TOOLS = {"add": add, "now": now}   # 注册表: 工具名 -> 可调用对象

# 分发调用：LLM 说 "调 add, 参数 a=2, b=3"，框架查表执行
print("add(2, 3) ->", TOOLS["add"](2, 3))
print("now()     ->", TOOLS["now"]())
print("已注册工具:", sorted(TOOLS))

add(2, 3) -> 5
now()     -> 2026-01-01 00:00:00 (固定字符串，保证 notebook 可复现)
已注册工具: ['add', 'now']


## ✏️ 练习 1：`run_sandboxed` —— 不抛异常的沙箱执行器

上面冒烟测试里超时会**抛异常**。但在评测 harness 里，"超时"是常态结果而不是程序错误 ——
我们希望它和正常返回走同一条路径，统一记进轨迹。

**任务**：实现 `run_sandboxed(cmd, timeout)`，返回四元组 `(stdout, stderr, returncode, timed_out)`：

- 正常结束：返回解码后的 `stdout` / `stderr`（str）、进程的 `returncode`、`timed_out=False`；
- 超时：**不向外抛异常**，捕获 `subprocess.TimeoutExpired`，返回 `returncode=None`、`timed_out=True`
  （stdout / stderr 给已捕获的部分，没有就空字符串）。

**提示**：
- `subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)`；
- 注意已知坑：`TimeoutExpired.stdout` / `.stderr` 可能是 `bytes` 也可能是 `None`，
  统一转成 str（`b.decode(errors="replace")`）；
- 10 行左右即可完成。

In [5]:
import subprocess

def run_sandboxed(cmd: list, timeout: float):
    '''在子进程沙箱中执行 cmd，永不向外抛超时异常。

    返回 (stdout: str, stderr: str, returncode: int | None, timed_out: bool)
    '''
    # TODO: 1) try: subprocess.run(..., capture_output=True, text=True, timeout=timeout)
    #          返回 (p.stdout, p.stderr, p.returncode, False)
    # TODO: 2) except subprocess.TimeoutExpired as e:
    #          把 e.stdout / e.stderr 规整成 str（None -> ""，bytes -> decode），
    #          返回 (out, err, None, True)
    raise NotImplementedError("完成 TODO 后删除此行")

In [ ]:
# ── 练习 1 自测：全过则达标（用 sys.executable -c，跨平台）──
import sys

# 用例 1：正常命令
out, err, rc, to = run_sandboxed([sys.executable, "-c", "print('hello')"], timeout=10)
assert to is False,        "正常命令不应标记超时"
assert rc == 0,            f"returncode 应为 0，得到 {rc}"
assert "hello" in out,     f"stdout 应含 'hello'，得到 {out!r}"

# 用例 2：非零退出码 + stderr（失败 != 超时）
out, err, rc, to = run_sandboxed(
    [sys.executable, "-c", "import sys; sys.stderr.write('boom'); sys.exit(3)"], timeout=10)
assert to is False and rc == 3, f"应正常返回 rc=3，得到 rc={rc}, timed_out={to}"
assert "boom" in err,           f"stderr 应含 'boom'，得到 {err!r}"

# 用例 3：超时 —— 子进程 sleep 10 秒，只给 1 秒，必须不抛异常
out, err, rc, to = run_sandboxed(
    [sys.executable, "-c", "import time; time.sleep(10)"], timeout=1.0)
assert to is True,   "超时必须标记 timed_out=True"
assert rc is None,   f"超时时 returncode 应为 None，得到 {rc}"
assert isinstance(out, str) and isinstance(err, str), "stdout/stderr 必须规整为 str"

print("✅ 练习 1 通过")

## ✏️ 练习 2：`dispatch` —— 带友好错误的工具分发

LLM 是会犯错的：拼错工具名是最常见的失败之一（小模型尤甚）。
agent 框架此时**不能崩溃**，而要把一条**友好的错误信息**作为 observation 回灌给模型，让它自己纠正 —— 这是模块 01/02 错误回灌机制的雏形。

**任务**：实现 `dispatch(tool_name, args, registry)`：

- `tool_name` 在 `registry` 中：调用 `registry[tool_name](**args)` 并返回结果；
- 不在：**不抛异常**，返回一条以 `"Error:"` 开头的字符串，需包含**未知工具名**和**当前可用工具列表**（如
  `"Error: unknown tool 'sub'. Available tools: ['add', 'now']"`），方便模型自我纠正。

**提示**：5 行左右；可用列表用 `sorted(registry)` 保证顺序稳定。

In [ ]:
def dispatch(tool_name: str, args: dict, registry: dict):
    '''按名字分发工具调用；未知工具返回友好错误字符串（不抛异常）。'''
    # TODO: 1) 若 tool_name 不在 registry：返回
    #          f"Error: unknown tool '{tool_name}'. Available tools: {sorted(registry)}"
    # TODO: 2) 否则返回 registry[tool_name](**args)
    raise NotImplementedError("完成 TODO 后删除此行")

In [ ]:
# ── 练习 2 自测：复用上面的 TOOLS 注册表 ──

# 用例 1：正常分发（带关键字参数）
assert dispatch("add", {"a": 2, "b": 3}, TOOLS) == 5

# 用例 2：无参工具
assert dispatch("now", {}, TOOLS) == NOW_FIXED

# 用例 3：未知工具 —— 友好错误而不是异常
msg = dispatch("sub", {"a": 1, "b": 2}, TOOLS)
assert isinstance(msg, str) and msg.startswith("Error:"), f"应返回 Error 开头的 str，得到 {msg!r}"
assert "sub" in msg,                "错误信息应包含未知工具名"
assert "add" in msg and "now" in msg, "错误信息应列出当前可用工具"

# 用例 4：空注册表（边界）
msg = dispatch("add", {}, {})
assert isinstance(msg, str) and msg.startswith("Error:")

print("✅ 练习 2 通过")

---
## 📖 参考答案

先自己做，再对照。两题合计不到 20 行 —— 但它们就是模块 03 的 harness 与模块 01 的 registry 的种子。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
import subprocess

def run_sandboxed(cmd: list, timeout: float):
    '''在子进程沙箱中执行 cmd，永不向外抛超时异常。'''
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.stdout, p.stderr, p.returncode, False
    except subprocess.TimeoutExpired as e:
        def _to_str(x):
            if x is None:
                return ""
            return x.decode(errors="replace") if isinstance(x, bytes) else x
        return _to_str(e.stdout), _to_str(e.stderr), None, True

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）

def dispatch(tool_name: str, args: dict, registry: dict):
    '''按名字分发工具调用；未知工具返回友好错误字符串（不抛异常）。'''
    if tool_name not in registry:
        return f"Error: unknown tool '{tool_name}'. Available tools: {sorted(registry)}"
    return registry[tool_name](**args)

---
## 各模块算力需求一览

| 模块 | 算力 | 说明 |
|---|---|---|
| 00 总览与环境 | CPU | 本 notebook，秒级 |
| 01 Tool Use | CPU（API 可选） | 纯 Python registry + 解析；有 key 可连真模型 |
| 02 ReAct Agent | CPU（GPU/MPS 加速可选） | Qwen2.5-1.5B 本地推理，CPU 单步约几十秒 |
| 03 沙箱与 harness | CPU | subprocess + 纯 Python，无模型 |
| 04 编码 Agent | CPU（GPU 可选） | pytest 跑 fail-to-pass；模型部分同 02 |
| 05 Computer Use | **GPU 建议（约 8GB）** | Qwen2-VL-2B 视觉推理；无 GPU 可只读代码 |
| 06 Agentic 评测 ★ | CPU（GPU 可选） | 统计与模拟为主（numpy） |
| 07 多智能体 | CPU（API 可选） | 编排逻辑纯 Python |
| 08 Agent 安全 ★ | CPU（GPU 可选） | toy 环境 + 监控规则，模型部分同 02 |

## ✅ 达标检查

- [ ] 依赖自检全绿（或明确知道缺什么、为什么暂时不装）
- [ ] 知道自己的 device 和 API key 配置状态
- [ ] 沙箱冒烟测试 + 两道练习的 `assert` 全部通过

**下一步 → 模块 01**：[`../01_tool_use/01_讲解.html`](../01_tool_use/01_讲解.html) ——
把今天 5 行的 registry 升级成带 JSON-schema 的正式 function calling，给 agent 一双可靠的手。